# DEM contour tutorial

Generate contour isobands from a synthetic DEM using `contourrs.contours_arrow`
and visualize them with GeoPandas + Matplotlib.

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from contourrs import contours_arrow

matplotlib.use("Agg")

## 1) Create a synthetic DEM

Sum of three Gaussian peaks to simulate a mountain range.

In [ ]:
size = 256
y, x = np.mgrid[-3 : 3 : complex(size), -3 : 3 : complex(size)]
dem = (
    np.exp(-(x**2 + y**2))
    + 0.7 * np.exp(-((x - 1.5) ** 2 + (y - 1) ** 2) / 0.5)
    + 0.5 * np.exp(-((x + 1.5) ** 2 + (y + 1.5) ** 2) / 0.8)
).astype(np.float32)

print(f"DEM shape: {dem.shape}, range: [{dem.min():.3f}, {dem.max():.3f}]")

## 2) Extract contour isobands

Define elevation thresholds and use `contours_arrow` to produce
filled contour polygons (isobands) as an Arrow table.
Convert to a GeoDataFrame for easy plotting.

In [ ]:
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1]

table = contours_arrow(dem, thresholds=thresholds)
gdf = gpd.GeoDataFrame.from_arrow(table)

print(f"Isobands: {len(gdf)} polygons")
print(f"Threshold values: {sorted(gdf['value'].unique())}")
gdf.head()

## 3) Plot: raster vs contour polygons

Side-by-side comparison of the continuous DEM and the filled contour bands.

In [ ]:
h, w = dem.shape

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: continuous DEM
axes[0].imshow(dem, cmap="terrain", interpolation="bilinear", extent=(0, w, h, 0))
axes[0].set_title("Synthetic DEM")
axes[0].set_axis_off()
axes[0].set_xlim(0, w)
axes[0].set_ylim(h, 0)
axes[0].set_aspect("equal")

# Right: filled contour polygons
gdf.plot(
    ax=axes[1],
    column="value",
    cmap="terrain",
    edgecolor="black",
    linewidth=0.3,
    legend=True,
    legend_kwds={"label": "Threshold", "shrink": 0.8},
)
axes[1].set_title(f"Isobands ({len(gdf)} polygons)")
axes[1].set_axis_off()
axes[1].set_xlim(0, w)
axes[1].set_ylim(h, 0)
axes[1].set_aspect("equal")

plt.tight_layout()